<a href="https://colab.research.google.com/github/kimmy111-zhu/human-validation/blob/main/notebooks/IFEval_matched_stratified_sample_typo%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import json
import random
import os
import pandas as pd
from google.colab import files


# =========================================================
# 1. Settings
# =========================================================

RANDOM_SEED = 42

# 抽取 5 个共同匹配成功的原始 prompt
# 每个 prompt 保留 0.1、0.4、0.7 三个版本
# 最终输出：5 × 3 = 15 行
SAMPLE_SIZE = 5

BENCHMARK_NAME = "IFEval"

TYPO_RATES = ["0.1", "0.4", "0.7"]


# =========================================================
# 2. File paths
# =========================================================

FILE_PATHS = {
    "0.1": "/content/raw_0.1.jsonl",
    "0.4": "/content/raw_0.4.jsonl",
    "0.7": "/content/raw_0.7.jsonl"
}


# Check whether all files exist
print("Checking input files:\n")

for typo_rate, file_path in FILE_PATHS.items():

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"File not found: {file_path}\n"
            f"Please upload raw_{typo_rate}.jsonl to Colab."
        )

    print(f"Rate {typo_rate}: {file_path}")


# =========================================================
# 3. Helper functions
# =========================================================

def read_jsonl(file_path):
    """
    Read a JSONL file and preserve the original row number.
    """

    records = []

    with open(file_path, "r", encoding="utf-8-sig") as file:

        for line_number, line in enumerate(file, start=1):

            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON on line {line_number}\n"
                    f"File: {file_path}\n"
                    f"Error: {error}"
                )

            record["_source_row"] = line_number
            records.append(record)

    return records


def normalize_text(value):
    """
    Normalize spaces and line breaks only for matching.

    The original text shown in the final CSV is not changed.
    """

    if value is None:
        return ""

    if isinstance(value, (dict, list)):
        value = json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True
        )

    return " ".join(str(value).split())


def convert_to_cell(value):
    """
    Convert lists and dictionaries into readable CSV text.
    """

    if value is None:
        return ""

    if isinstance(value, (dict, list)):
        return json.dumps(
            value,
            ensure_ascii=False
        )

    return str(value)


def get_first_available(record, possible_fields):
    """
    Return the first non-empty field found in the record.
    """

    for field_name in possible_fields:

        value = record.get(field_name)

        if value is None:
            continue

        if isinstance(value, (dict, list)):
            if len(value) > 0:
                return value

        elif str(value).strip() != "":
            return value

    return ""


def get_original_prompt(record):
    """
    Extract the original prompt used to match records
    across the three typo rates.
    """

    return get_first_available(
        record,
        [
            "original_text_backup",
            "original_prompt",
            "original_instruction",
            "original_text",
            "prompt_original"
        ]
    )


def get_modified_prompt(record):
    """
    Extract the typo-transformed IFEval prompt.
    """

    return get_first_available(
        record,
        [
            "prompt",
            "modified_prompt",
            "instruction",
            "question",
            "text",
            "input"
        ]
    )


def get_gold_answer(record):
    """
    Extract the reference answer if present.

    IFEval may not have a conventional gold answer,
    so empty values may be normal.
    """

    return get_first_available(
        record,
        [
            "answer",
            "output",
            "reference_answer",
            "gold_answer",
            "reference",
            "response"
        ]
    )


def get_instruction_ids(record):
    """
    Extract IFEval instruction or constraint IDs.
    """

    return get_first_available(
        record,
        [
            "instruction_id_list",
            "instruction_ids",
            "instruction_id",
            "constraint_ids"
        ]
    )


def get_instruction_kwargs(record):
    """
    Extract the parameters associated with IFEval constraints.
    """

    return get_first_available(
        record,
        [
            "kwargs",
            "instruction_kwargs",
            "constraint_kwargs",
            "arguments"
        ]
    )


# =========================================================
# 4. Read the three JSONL files
# =========================================================

datasets = {}

print("\nLoading files:\n")

for typo_rate, file_path in FILE_PATHS.items():

    datasets[typo_rate] = read_jsonl(file_path)

    print(
        f"Loaded {len(datasets[typo_rate])} records "
        f"for typo rate {typo_rate}"
    )


# Print field names for checking
print("\nFields found in the first record of each file:")

for typo_rate in TYPO_RATES:

    if datasets[typo_rate]:

        print(
            f"\nRate {typo_rate}: "
            f"{list(datasets[typo_rate][0].keys())}"
        )


# =========================================================
# 5. Create matching maps
# =========================================================

record_maps = {}

print("\nBuilding matching maps:")

for typo_rate, records in datasets.items():

    current_map = {}

    missing_original_count = 0
    duplicate_count = 0

    for record in records:

        original_prompt = get_original_prompt(record)

        normalized_original = normalize_text(
            original_prompt
        )

        if not normalized_original:
            missing_original_count += 1
            continue

        if normalized_original in current_map:
            duplicate_count += 1
            continue

        current_map[normalized_original] = record

    record_maps[typo_rate] = current_map

    print(
        f"\nRate {typo_rate}: "
        f"{len(current_map)} usable original prompts"
    )

    if missing_original_count > 0:
        print(
            f"Warning: {missing_original_count} records "
            f"had no original prompt."
        )

    if duplicate_count > 0:
        print(
            f"Warning: {duplicate_count} duplicate original prompts "
            f"were found. The first record was kept."
        )


# =========================================================
# 6. Find prompts shared by all three typo-rate files
# =========================================================

common_original_keys = set(
    record_maps["0.1"].keys()
)

for typo_rate in ["0.4", "0.7"]:

    common_original_keys &= set(
        record_maps[typo_rate].keys()
    )


# Sort before sampling to make the process fully reproducible
common_original_keys = sorted(
    common_original_keys
)


print(
    f"\nCommon matched original prompts: "
    f"{len(common_original_keys)}"
)


if len(common_original_keys) < SAMPLE_SIZE:

    raise ValueError(
        f"Only {len(common_original_keys)} matched prompts "
        f"were found across all three files.\n"
        f"Cannot sample {SAMPLE_SIZE} prompts."
    )


# =========================================================
# 7. Randomly sample 5 matched original prompts
# =========================================================

random.seed(RANDOM_SEED)

selected_original_keys = random.sample(
    common_original_keys,
    SAMPLE_SIZE
)


print(f"\nRandom seed: {RANDOM_SEED}")

print(
    f"Selected matched prompts: "
    f"{len(selected_original_keys)}"
)


# =========================================================
# 8. Create matched stratified sample
# =========================================================

output_rows = []


for prompt_number, original_key in enumerate(
    selected_original_keys,
    start=1
):

    base_id = (
        f"{BENCHMARK_NAME}_{prompt_number:03d}"
    )

    # Use the original prompt from the rate 0.1 record
    reference_record = record_maps["0.1"][
        original_key
    ]

    original_prompt = get_original_prompt(
        reference_record
    )

    for typo_rate in TYPO_RATES:

        record = record_maps[typo_rate][
            original_key
        ]

        modified_prompt = get_modified_prompt(
            record
        )

        gold_answer = get_gold_answer(
            record
        )

        instruction_ids = get_instruction_ids(
            record
        )

        instruction_kwargs = get_instruction_kwargs(
            record
        )

        output_rows.append({

            "Base_ID": base_id,

            "Sample_ID": (
                f"{base_id}_rate_{typo_rate}"
            ),

            "Benchmark": BENCHMARK_NAME,

            "Typo_Rate": typo_rate,

            "Source_File": os.path.basename(
                FILE_PATHS[typo_rate]
            ),

            "Source_Row": record.get(
                "_source_row",
                ""
            ),

            "Original_Prompt": convert_to_cell(
                original_prompt
            ),

            "Modified_Prompt": convert_to_cell(
                modified_prompt
            ),

            "Gold_Answer": convert_to_cell(
                gold_answer
            ),

            "Instruction_IDs": convert_to_cell(
                instruction_ids
            ),

            "Instruction_Kwargs": convert_to_cell(
                instruction_kwargs
            ),

            # Reviewer 1
            "R1_Meaning": "",
            "R1_Key_Info": "",
            "R1_Answer_Preserved": "",
            "R1_Realism": "",
            "R1_Readability": "",
            "R1_Comments": "",

            # Reviewer 2
            "R2_Meaning": "",
            "R2_Key_Info": "",
            "R2_Answer_Preserved": "",
            "R2_Realism": "",
            "R2_Readability": "",
            "R2_Comments": "",

            # Final adjudicated result
            "Final_Meaning": "",
            "Final_Key_Info": "",
            "Final_Answer_Preserved": "",
            "Final_Realism": "",
            "Final_Readability": "",
            "Final_Comments": ""
        })


# =========================================================
# 9. Convert to DataFrame
# =========================================================

sample_df = pd.DataFrame(
    output_rows
)


print("\nSampling completed.")

print(
    f"Selected original prompts: "
    f"{SAMPLE_SIZE}"
)

print(
    f"Number of typo-rate strata: "
    f"{len(TYPO_RATES)}"
)

print(
    f"Total output rows: "
    f"{len(sample_df)}"
)


display(sample_df)


# =========================================================
# 10. Validate the final sample
# =========================================================

empty_original = (
    sample_df["Original_Prompt"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_modified = (
    sample_df["Modified_Prompt"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_gold_answer = (
    sample_df["Gold_Answer"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_instruction_ids = (
    sample_df["Instruction_IDs"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)


print("\nColumn check:")

print(
    f"Empty Original_Prompt rows: "
    f"{empty_original}"
)

print(
    f"Empty Modified_Prompt rows: "
    f"{empty_modified}"
)

print(
    f"Empty Gold_Answer rows: "
    f"{empty_gold_answer}"
)

print(
    f"Empty Instruction_IDs rows: "
    f"{empty_instruction_ids}"
)


# Check the number of records in each typo-rate stratum
print("\nRows per typo-rate stratum:")

print(
    sample_df["Typo_Rate"]
    .value_counts()
    .sort_index()
)


# Check that each Base_ID has exactly three versions
base_id_counts = (
    sample_df
    .groupby("Base_ID")
    .size()
)

incorrect_base_ids = base_id_counts[
    base_id_counts != len(TYPO_RATES)
]


if len(incorrect_base_ids) == 0:

    print(
        "\nMatched sampling check passed: "
        "every Base_ID has 3 typo-rate versions."
    )

else:

    print(
        "\nWarning: Some Base_ID values do not have "
        "exactly 3 typo-rate versions:"
    )

    print(incorrect_base_ids)


if empty_original > 0:

    print(
        "\nWarning: Some Original_Prompt values are empty."
    )


if empty_modified > 0:

    print(
        "\nWarning: Some Modified_Prompt values are empty.\n"
        "Check the field names printed near the beginning "
        "of the output."
    )


if empty_gold_answer > 0:

    print(
        "\nNote: Empty Gold_Answer values can be normal "
        "for IFEval because it evaluates instruction "
        "following rather than a conventional gold answer."
    )


if empty_instruction_ids > 0:

    print(
        "\nNote: Some Instruction_IDs values are empty.\n"
        "Check whether the source file uses another "
        "field name for its instruction constraints."
    )


# =========================================================
# 11. Save and download CSV
# =========================================================

output_file = (
    "/content/"
    "IFEval_matched_stratified_sample_n5_seed42.csv"
)


sample_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"\nCSV saved successfully: "
    f"{output_file}"
)


files.download(output_file)

Checking input files:

Rate 0.1: /content/raw_0.1.jsonl
Rate 0.4: /content/raw_0.4.jsonl
Rate 0.7: /content/raw_0.7.jsonl

Loading files:

Loaded 805 records for typo rate 0.1
Loaded 805 records for typo rate 0.4
Loaded 805 records for typo rate 0.7

Fields found in the first record of each file:

Rate 0.1: ['datasplit', 'dataset', 'instruction', 'input', 'output', 'generator', 'sample_mode', 'typo_rate_metadata', 'original_text_backup', 'typo_source_field', '_source_row']

Rate 0.4: ['datasplit', 'dataset', 'instruction', 'input', 'output', 'generator', 'sample_mode', 'typo_rate_metadata', 'original_text_backup', 'typo_source_field', '_source_row']

Rate 0.7: ['datasplit', 'dataset', 'instruction', 'input', 'output', 'generator', 'sample_mode', 'typo_rate_metadata', 'original_text_backup', 'typo_source_field', '_source_row']

Building matching maps:

Rate 0.1: 804 usable original prompts

Rate 0.4: 804 usable original prompts

Rate 0.7: 804 usable original prompts

Common matched orig

,Base_ID,Sample_ID,Benchmark,Typo_Rate,Source_File,Source_Row,Original_Prompt,Modified_Prompt,Gold_Answer,Instruction_IDs,...,R2_Answer_Preserved,R2_Realism,R2_Readability,R2_Comments,Final_Meaning,Final_Key_Info,Final_Answer_Preserved,Final_Realism,Final_Readability,Final_Comments
0,IFEval_001,IFEval_001_rate_0.1,IFEval,0.1,raw_0.1.jsonl,285,Write a detailed patent writing for an innovat...,Wrtie a detailed patent writing for an innovat...,This patent writing details an innovative and ...,,...,,,,,,,,,,
1,IFEval_001,IFEval_001_rate_0.4,IFEval,0.4,raw_0.4.jsonl,285,Write a detailed patent writing for an innovat...,Writre a detailed patent writng for am innovti...,This patent writing details an innovative and ...,,...,,,,,,,,,,
2,IFEval_001,IFEval_001_rate_0.7,IFEval,0.7,raw_0.7.jsonl,285,Write a detailed patent writing for an innovat...,Write a detaile patent writingh frt an innovat...,This patent writing details an innovative and ...,,...,,,,,,,,,,
3,IFEval_002,IFEval_002_rate_0.1,IFEval,0.1,raw_0.1.jsonl,566,Design a programming problem related to the su...,Design a programming problem related to the su...,Design a programming problem related to Dynami...,,...,,,,,,,,,,
4,IFEval_002,IFEval_002_rate_0.4,IFEval,0.4,raw_0.4.jsonl,566,Design a programming problem related to the su...,Design a programmign problem relatewd to the s...,Design a programming problem related to Dynami...,,...,,,,,,,,,,
5,IFEval_002,IFEval_002_rate_0.7,IFEval,0.7,raw_0.7.jsonl,566,Design a programming problem related to the su...,Dwesig a programming problen relatd to tjer su...,Design a programming problem related to Dynami...,,...,,,,,,,,,,
6,IFEval_003,IFEval_003_rate_0.1,IFEval,0.1,raw_0.1.jsonl,754,"As a space colonist on Mars, describe your dai...","As a space clonist on Mars, describe yoiur dai...","As a space colonist on Mars, my daily life is ...",,...,,,,,,,,,,
7,IFEval_003,IFEval_003_rate_0.4,IFEval,0.4,raw_0.4.jsonl,754,"As a space colonist on Mars, describe your dai...","A a sacr colonist on Mars, descrbie your daily...","As a space colonist on Mars, my daily life is ...",,...,,,,,,,,,,
8,IFEval_003,IFEval_003_rate_0.7,IFEval,0.7,raw_0.7.jsonl,754,"As a space colonist on Mars, describe your dai...","As a spce colonist ob Msrs, descrbie yourt dai...","As a space colonist on Mars, my daily life is ...",,...,,,,,,,,,,
9,IFEval_004,IFEval_004_rate_0.1,IFEval,0.1,raw_0.1.jsonl,212,is queue an ADT or a data structure,is queue an AD or a data structure,Queue is both an ADT (Abstract Data Type) and ...,,...,,,,,,,,,,



Column check:
Empty Original_Prompt rows: 0
Empty Modified_Prompt rows: 0
Empty Gold_Answer rows: 0
Empty Instruction_IDs rows: 15

Rows per typo-rate stratum:
Typo_Rate
0.1    5
0.4    5
0.7    5
Name: count, dtype: int64

Matched sampling check passed: every Base_ID has 3 typo-rate versions.

Note: Some Instruction_IDs values are empty.
Check whether the source file uses another field name for its instruction constraints.

CSV saved successfully: /content/IFEval_matched_stratified_sample_n5_seed42.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>